# Healthcare Claims Fraud Detection

## Model Insights and Business Recommendations

This notebook translates machine learning results into actionable business insights for healthcare fraud detection.

The objective is to identify the key drivers of fraudulent provider behavior, evaluate the impact of fraud detection, and provide recommendations that can support fraud investigation teams, healthcare administrators, and insurance organizations.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")
print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Provider-Level Dataset

In [3]:
provider_df = pd.read_csv("../data/processed/provider_fraud_modeling_dataset.csv")
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


## Dataset Overview

In [4]:
print(provider_df.shape)
provider_df.columns.tolist()

(5410, 20)


['Provider',
 'PotentialFraud',
 'InpatientClaimCount',
 'UniqueInpatientBeneficiaries',
 'AvgInpatientReimbursement',
 'TotalInpatientReimbursement',
 'AvgLengthOfStay',
 'MaxLengthOfStay',
 'AvgDiagnosisCount',
 'OutpatientClaimCount',
 'UniqueOutpatientBeneficiaries',
 'AvgOutpatientReimbursement',
 'TotalOutpatientReimbursement',
 'AvgPatientAge',
 'AvgChronicConditions',
 'DeceasedRate',
 'AvgIPAnnualReimbursement',
 'AvgOPAnnualReimbursement',
 'UniqueBeneficiaries',
 'Fraud']

### Interpretation

The dataset contains a comprehensive set of provider-level metrics covering healthcare utilization, reimbursement patterns, beneficiary characteristics, and fraud outcomes.

These variables provide multiple perspectives for identifying suspicious provider behavior and understanding the operational characteristics associated with healthcare fraud.

## Fraud Distribution Analysis

In [5]:
fraud_distribution = provider_df["PotentialFraud"].value_counts()
fraud_distribution

PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64

### Interpretation

The dataset contains 5,410 healthcare providers, of which:

- 4,904 providers (90.65%) are labeled as Non-Fraudulent
- 506 providers (9.35%) are labeled as Fraudulent

This indicates that fraudulent providers represent a relatively small portion of the overall provider population. Such class imbalance is common in fraud detection problems and highlights the importance of specialized techniques, such as SMOTE, during model development.

From a business perspective, even a small percentage of fraudulent providers can result in substantial financial losses due to the high volume and value of healthcare claims processed.

## Fraud vs Non-Fraud Provider Distribution

In [7]:
fig = px.pie(
    provider_df,
    names="PotentialFraud",
    title="Fraud vs Non-Fraud Provider Distribution"
)
fig.update_layout(width=800, height=600)
fig.show()

### Interpretation

The provider population is highly imbalanced, with non-fraudulent providers representing approximately 91% of all providers and fraudulent providers accounting for about 9%.

Although fraudulent providers are relatively rare, they may contribute disproportionately to healthcare costs and reimbursement expenditures. This imbalance highlights the importance of effective fraud detection systems that can identify high-risk providers without generating excessive false alarms.

## Business Insight 1: Fraudulent Providers Generate Higher Reimbursements

In [8]:
reimbursement_summary = provider_df.groupby(
    "PotentialFraud"
)[[
    "TotalInpatientReimbursement",
    "TotalOutpatientReimbursement",
    "AvgInpatientReimbursement",
    "AvgOutpatientReimbursement"
]].mean().round(2)
reimbursement_summary

,TotalInpatientReimbursement,TotalOutpatientReimbursement,AvgInpatientReimbursement,AvgOutpatientReimbursement
PotentialFraud,,,,
No,34055.57,19138.15,3363.58,262.22
Yes,476854.76,107495.28,9605.43,473.62


### Interpretation

Fraudulent providers generate substantially higher reimbursement amounts than non-fraudulent providers across both inpatient and outpatient services.

Key observations:

- Average Total Inpatient Reimbursement:
  - Non-Fraud: $34,056
  - Fraud: $476,855

- Average Total Outpatient Reimbursement:
  - Non-Fraud: $19,138
  - Fraud: $107,495

- Average Inpatient Reimbursement per Claim:
  - Non-Fraud: $3,364
  - Fraud: $9,605

- Average Outpatient Reimbursement per Claim:
  - Non-Fraud: $262
  - Fraud: $474

Fraudulent providers receive significantly larger reimbursements, particularly for inpatient services. This suggests that excessive billing and unusually large claim amounts are among the strongest indicators of potential fraud.

Healthcare organizations should prioritize providers with exceptionally high reimbursement volumes for investigation and audit activities.

## Reimbursement Comparison: Fraud vs Non-Fraud Providers

In [9]:
comparison_df = reimbursement_summary.reset_index()
fig = px.bar(
    comparison_df,
    x="PotentialFraud",
    y="TotalInpatientReimbursement",
    color="PotentialFraud",
    title="Average Total Inpatient Reimbursement by Provider Type",
    text="TotalInpatientReimbursement"
)
fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)
fig.update_layout(width=900, height=600, showlegend=False)
fig.show()

### Interpretation

The reimbursement comparison reveals a substantial financial difference between fraudulent and non-fraudulent providers.

Fraudulent providers receive approximately fourteen times more inpatient reimbursement than non-fraudulent providers.

This finding suggests that unusually high reimbursement activity is one of the strongest indicators of potential fraud. Healthcare organizations can leverage reimbursement-based monitoring systems to identify providers that warrant further investigation.

## Business Insight 2: Fraudulent Providers Serve More Beneficiaries and Submit More Claims

In [10]:
claim_summary = provider_df.groupby(
    "PotentialFraud"
)[[
    "InpatientClaimCount",
    "OutpatientClaimCount",
    "UniqueBeneficiaries",
    "UniqueInpatientBeneficiaries",
    "UniqueOutpatientBeneficiaries"
]].mean().round(2)
claim_summary

,InpatientClaimCount,OutpatientClaimCount,UniqueBeneficiaries,UniqueInpatientBeneficiaries,UniqueOutpatientBeneficiaries
PotentialFraud,,,,,
No,3.48,66.95,49.11,3.26,46.12
Yes,46.25,374.30,242.02,40.76,206.77


### Interpretation

Fraudulent providers serve significantly more beneficiaries and submit substantially more healthcare claims than non-fraudulent providers.

Key observations:

- Average Inpatient Claims:
  - Non-Fraud: 3.48
  - Fraud: 46.25

- Average Outpatient Claims:
  - Non-Fraud: 66.95
  - Fraud: 374.30

- Average Unique Beneficiaries:
  - Non-Fraud: 49.11
  - Fraud: 242.02

- Average Unique Inpatient Beneficiaries:
  - Non-Fraud: 3.26
  - Fraud: 40.76

Fraudulent providers consistently demonstrate higher healthcare utilization volumes across multiple dimensions.

These results suggest that unusually high patient volume and claim activity may serve as early warning indicators for potential fraud investigations.

## Claims and Beneficiary Volume Comparison

In [11]:
fig = px.bar(
    claim_summary.reset_index(),
    x="PotentialFraud",
    y="UniqueBeneficiaries",
    color="PotentialFraud",
    title="Average Unique Beneficiaries by Provider Type",
    text="UniqueBeneficiaries"
)
fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)
fig.update_layout(width=900, height=600, showlegend=False)
fig.show()

### Interpretation

The beneficiary comparison reveals that fraudulent providers serve nearly five times more beneficiaries than non-fraudulent providers.

Higher beneficiary volume may increase opportunities for excessive billing, duplicate claims, unnecessary procedures, or reimbursement manipulation.

Healthcare organizations should monitor providers with unusually large beneficiary populations, particularly when combined with elevated reimbursement levels and claim volumes.

## Business Insight 3: Fraudulent Providers Exhibit Longer Lengths of Stay

In [12]:
los_summary = provider_df.groupby("PotentialFraud")[[
    "AvgLengthOfStay",
    "MaxLengthOfStay"
]].mean().round(2)
los_summary

,AvgLengthOfStay,MaxLengthOfStay
PotentialFraud,,
No,1.87,4.48
Yes,5.32,23.75


### Interpretation

Fraudulent providers exhibit substantially longer patient stay durations compared to non-fraudulent providers.

Key observations:

- Average Length of Stay:
  - Non-Fraud: 1.87 days
  - Fraud: 5.32 days

- Maximum Length of Stay:
  - Non-Fraud: 4.48 days
  - Fraud: 23.75 days

Longer patient stays may indicate excessive treatment durations, unnecessary admissions, or abnormal billing practices designed to increase reimbursement amounts.

Because length-of-stay metrics are strongly associated with provider fraud, healthcare organizations should establish monitoring thresholds for providers with unusually long inpatient stays.

## Length of Stay Comparison

In [13]:
fig = px.bar(
    los_summary.reset_index(),
    x="PotentialFraud",
    y="MaxLengthOfStay",
    color="PotentialFraud",
    title="Average Maximum Length of Stay by Provider Type",
    text="MaxLengthOfStay")
fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside")
fig.update_layout(width=900, height=600, showlegend=False)
fig.show()

### Interpretation

The visualization highlights a significant difference in maximum patient stay duration between fraudulent and non-fraudulent providers.

Fraudulent providers report an average maximum stay of approximately 24 days, compared to less than 5 days for non-fraudulent providers.

Extended lengths of stay may indicate unnecessary admissions, prolonged treatment periods, or billing practices designed to maximize reimbursement revenue.

Length-of-stay monitoring should therefore be incorporated into healthcare fraud surveillance programs as a key risk indicator.

## Executive Recommendations

### Recommendation 1: Prioritize High-Reimbursement Providers

Providers with exceptionally high inpatient reimbursement totals should be reviewed regularly through automated audit programs.

The analysis shows that fraudulent providers generate substantially larger reimbursement amounts than non-fraudulent providers, making reimbursement activity one of the strongest fraud indicators.

### Recommendation 2: Monitor Unusually Long Patient Stays

Length-of-stay metrics should be incorporated into fraud monitoring systems.

Providers exhibiting unusually high average or maximum stay durations should be flagged for further investigation because prolonged admissions may indicate unnecessary services or reimbursement manipulation.

### Recommendation 3: Establish Provider Risk Scoring

Healthcare organizations should develop provider risk scores using variables such as:

- Total Inpatient Reimbursement
- Inpatient Claim Count
- Unique Beneficiaries
- Length of Stay Metrics
- Average Reimbursement Amounts

Risk-based monitoring enables fraud investigators to focus resources on the providers most likely to engage in suspicious activity.

### Recommendation 4: Deploy Predictive Fraud Detection Models

The Logistic Regression model achieved:

- ROC-AUC: 96.99%
- Recall: 91.09%

The model successfully identified the majority of fraudulent providers and can serve as an effective early-warning system for healthcare fraud detection.

Deploying predictive analytics can significantly improve audit efficiency and reduce financial losses associated with fraudulent claims.

## Executive Summary

This project developed an end-to-end healthcare fraud detection solution using provider-level healthcare claims data.

Key findings revealed that fraudulent providers:

- Generate substantially higher reimbursement amounts
- Serve significantly more beneficiaries
- Submit larger volumes of claims
- Exhibit longer patient stay durations

Machine learning models successfully identified fraud-related patterns, with Logistic Regression emerging as the preferred model due to its strong fraud detection performance and high recall.

The results demonstrate that healthcare fraud can be detected effectively using claims-based provider analytics, enabling organizations to proactively identify high-risk providers and reduce financial losses.